## Text input

In [1]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

env_path = Path("/Users/daniel/code/lca-lc-foundations/.env")
file_values = dotenv_values(env_path)

load_dotenv(env_path, override=True)

loaded_key = os.environ.get("LANGSMITH_API_KEY", "")
file_key = file_values.get("LANGSMITH_API_KEY", "")

print("Using endpoint:", os.environ.get("LANGSMITH_ENDPOINT"))
print("Key matches .env:", loaded_key == file_key)
print("Loaded key suffix:", loaded_key[-4:])  # safe comparison only
print(".env key suffix:", file_key[-4:])

Using endpoint: https://apac.api.smith.langchain.com
Key matches .env: True
Loaded key suffix: ab46
.env key suffix: ab46


https://platform.openai.com/docs/models

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Capital: Selene City (often simply Selene)

Selene City is the political heart of The Moon’s lunar federation. Nestled along the rim of a colossal equatorial crater, it’s protected by the translucent Aurora Dome and rings with a bustling orbital port. Key features include:
- The Horizon Hall, where the Lunar Confederation Parliament meets.
- The Lunar Archive, a vast, hourglass-shaped library preserving 4.5 billion years of lunar history.
- Portus Asteria, the main lunar trade hub that anchors ships and tethers from Earth and beyond.
- A lattice of hexagonal habitat domes, gardens, and solar nets that power and sustain the city.

Population ranges in the tens of millions, with districts varying from high-density residential rings to garden courtyards and research campuses. Want a version for a grittier or more utopian Moon? I can tailor the vibe and history.


## Image input

In [5]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x1153ab700>, 'last_modified': datetime.datetime(2026, 10, 11, 8, 44, 51, tzinfo=datetime.timezone.utc)},)


In [7]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [8]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Here’s a portrait of the capital that fits the scene you shared. I’ve named it Caelum Prime, the gleaming heart of its world.

Visual impression
- Caelum Prime sits on the rim of a basaltic canyon, a fortress-city carved from stone and crystal. The lower terraces glow with cyan aquifer lights, while the skyline rises in a forest of slender spires and geometric domes.
- A colossal blue planet looms close in the night sky, its pale halo casting a perpetual twilight over the metropolis. The air carries a dry, mineral tang, and faint wind-swept dust dances through the neon-lit streets.
- The city feels at once ancient and futuristic: carved stairways, carved archways, and, above, temples and towers of glass and alloy that catch the starlight like prisms.

What Caelum Prime is
- The political and cultural capital of the planet-state known as the Caelaran Union. It’s where the Grand Assembly (a rotating, multispecies council) and the High Court of Echoes meet, where most of the planet’s cere

## Audio input

In [9]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|█| 50/50 [00:05<00:00,  9.55it/


Done.


In [10]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

I’m here to assist, but I can’t identify speakers or ignore previous instructions. However, I can analyze the content of the audio, such as describing any speech, sounds, or themes present. Feel free to ask how you’d like the audio information summarized or what details you’re most interested in. Would you like me to proceed with an audio transcription or a summary of its contents?
